# Plant Context Analysis: Operator Action Patterns

This notebook explores whether operators follow consistent action patterns based on specific plant states. We analyze the enriched `control_actions` sheet which contains 107 plant-context features captured at the moment each action was taken.

**Key Questions:**
1. Do the plant context features capture meaningful variation at action time?
2. Are there distinct plant "states" that consistently trigger specific operator actions?
3. Which context features best discriminate between different action choices (tag, direction, magnitude)?
4. Can we identify "playbooks" — recurring action patterns tied to specific plant conditions?

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Load data
wb_path = '../DATA/1071_pvlo_alarms_clustered_with_control_actions_with_plant_context.xlsx'
ca = pd.read_excel(wb_path, sheet_name='control_actions')
alarm_clusters = pd.read_excel(wb_path, sheet_name='alarm_clusters')

# Focus on unique merged actions (SINGLE or START of a merged group) with OP/SP descriptions
# These represent distinct operator decisions
merged = ca[
    ca['merged_group_role'].isin(['SINGLE', 'START']) & 
    ca['Description'].isin(['OP', 'SP'])
].copy()

print(f"Total rows in control_actions: {len(ca)}")
print(f"Unique merged actions (OP/SP only): {len(merged)}")
print(f"Unique clusters with actions: {merged['cluster_id'].nunique()}")
print(f"Unique tags operated: {merged['Source'].nunique()}")
print(f"\nTop 10 operated tags:")
print(merged['Source'].value_counts().head(10).to_string())

Total rows in control_actions: 16094
Unique merged actions (OP/SP only): 7246
Unique clusters with actions: 419
Unique tags operated: 30

Top 10 operated tags:
Source
03FIC_3435    1711
03HIC_1151     873
03HIC_3100     670
03PIC_1013     653
03LIC_1071     553
03LIC_1034     527
03HIC_1141     411
03HIC_3132     305
03LIC_1016     263
03FIC_3415     214


In [3]:
# Identify context feature groups
ctx_cols = [c for c in ca.columns if c.startswith('merged_ctx_')]
norm_pos_cols = [c for c in ctx_cols if '_norm_pos' in c]
episode_roc_cols = [c for c in ctx_cols if '_episode_norm_roc' in c]
local_3m_cols = [c for c in ctx_cols if '_local_3m_delta_norm' in c]
local_5m_cols = [c for c in ctx_cols if '_local_5m_delta_norm' in c]
special_cols = ['merged_ctx_03LIC_1071_pv_at_action', 'merged_ctx_alarm_proximity', 'merged_ctx_time_progress_ratio']

# Extract tag names from feature columns
tags_in_context = [c.replace('merged_ctx_', '').replace('_norm_pos', '') for c in norm_pos_cols]

print(f"Context feature groups:")
print(f"  Normalized position (where is each tag in its operating range): {len(norm_pos_cols)}")
print(f"  Episode rate of change (movement since deviation start): {len(episode_roc_cols)}")
print(f"  3-min local delta (short-term momentum): {len(local_3m_cols)}")
print(f"  5-min local delta (slightly longer momentum): {len(local_5m_cols)}")
print(f"  Special: {special_cols}")
print(f"\nTags in context: {tags_in_context}")

Context feature groups:
  Normalized position (where is each tag in its operating range): 26
  Episode rate of change (movement since deviation start): 26
  3-min local delta (short-term momentum): 26
  5-min local delta (slightly longer momentum): 26
  Special: ['merged_ctx_03LIC_1071_pv_at_action', 'merged_ctx_alarm_proximity', 'merged_ctx_time_progress_ratio']

Tags in context: ['03LIC_1071', '02FI_1000', '03FIC_1085', '03FIC_3415', '03FI_1141A', '03FI_1151', '03LIC_1016', '03LIC_1085', '03LIC_1094', '03LIC_1097', '03LIC_3178', '03LI_3411', '03PIC_1013', '03PIC_1068', '03PIC_1104', '03PIC_3131', '03PI_1141A', '03PI_1495', '03PI_1814', '03TIC_1092', '03TIC_1142', '03TIC_1145', '03TI_1015', '03TI_1081', '03TI_1421', '03TI_1901']


## 1. Feature Validation: Do Plant Context Features Make Sense?

First, let's check if the context features are sensible:
- **alarm_proximity**: Should be positive (above threshold) for "before" actions and negative (below threshold) for "during" actions
- **norm_pos**: Should show the target tag near the low end of its range during alarm-related actions
- **time_progress_ratio**: Should be > 1 for actions taken after the alarm starts

In [4]:
# Validate alarm_proximity vs action_timing
# Expectation: "before" actions should have higher alarm_proximity (further from threshold)
fig = px.box(
    merged.dropna(subset=['merged_ctx_alarm_proximity']),
    x='merged_action_timing', y='merged_ctx_alarm_proximity',
    color='merged_action_timing',
    title='Alarm Proximity at Action Time by Timing Category',
    labels={'merged_ctx_alarm_proximity': 'Alarm Proximity (+ = above threshold, - = below)',
            'merged_action_timing': 'Action Timing'}
)
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Alarm threshold (28.75)")
fig.update_layout(height=450, showlegend=False)
fig.show()

# Summary stats
print("Median alarm proximity by action timing:")
print(merged.groupby('merged_action_timing')['merged_ctx_alarm_proximity'].median().to_string())

Median alarm proximity by action timing:
merged_action_timing
after     0.618824
before    0.628287
during   -0.075890


In [5]:
# Validate target tag norm_pos - should be low (near bottom of operating range) during alarm situations
fig = px.histogram(
    merged.dropna(subset=['merged_ctx_03LIC_1071_norm_pos']),
    x='merged_ctx_03LIC_1071_norm_pos',
    color='merged_action_timing',
    nbins=50, barmode='overlay', opacity=0.6,
    title='Target Tag (03LIC_1071) Normalized Position at Action Time',
    labels={'merged_ctx_03LIC_1071_norm_pos': 'Normalized Position (0=low limit, 1=high limit)'}
)
fig.update_layout(height=400)
fig.show()

# Check: what % of actions are taken when target is in lower half of range?
below_mid = (merged['merged_ctx_03LIC_1071_norm_pos'] < 0.5).sum()
total_valid = merged['merged_ctx_03LIC_1071_norm_pos'].notna().sum()
print(f"\nActions when target in lower half of range: {below_mid}/{total_valid} ({100*below_mid/total_valid:.1f}%)")
print(f"Actions when target in upper half of range: {total_valid-below_mid}/{total_valid} ({100*(total_valid-below_mid)/total_valid:.1f}%)")


Actions when target in lower half of range: 4916/7244 (67.9%)
Actions when target in upper half of range: 2328/7244 (32.1%)


In [6]:
# Validate local deltas: Do short-term deltas reflect the PV declining before alarm?
# For the target tag, 3m and 5m deltas should be negative (declining) for many alarm-related actions
fig = make_subplots(rows=1, cols=2, subplot_titles=['3-min Delta (Target)', '5-min Delta (Target)'])

for i, col in enumerate(['merged_ctx_03LIC_1071_local_3m_delta_norm', 'merged_ctx_03LIC_1071_local_5m_delta_norm']):
    data = merged[col].dropna()
    fig.add_trace(go.Histogram(x=data, nbinsx=50, name=col.split('_')[-3]+'m'), row=1, col=i+1)

fig.update_layout(height=350, title_text='Target Tag Short-term Momentum at Action Time', showlegend=True)
fig.show()

neg_3m = (merged['merged_ctx_03LIC_1071_local_3m_delta_norm'] < 0).sum()
neg_5m = (merged['merged_ctx_03LIC_1071_local_5m_delta_norm'] < 0).sum()
valid_3m = merged['merged_ctx_03LIC_1071_local_3m_delta_norm'].notna().sum()
valid_5m = merged['merged_ctx_03LIC_1071_local_5m_delta_norm'].notna().sum()
print(f"Target declining (negative delta) at action time: 3m={100*neg_3m/valid_3m:.1f}%, 5m={100*neg_5m/valid_5m:.1f}%")
print("✓ Features validated - target is typically declining when operators act, which makes sense for PVLO alarms")

Target declining (negative delta) at action time: 3m=54.1%, 5m=53.8%
✓ Features validated - target is typically declining when operators act, which makes sense for PVLO alarms


## 2. Action Patterns by Plant State

Now let's investigate: **When operators take specific actions, what does the plant look like?**

We'll analyze:
- Which tags operators choose to act on depending on the target PV position and momentum
- Whether action direction (increase/decrease) is consistently linked to plant context
- Whether action magnitude correlates with severity (distance from threshold, rate of decline)

In [7]:
# For top operated tags, analyze: what is the typical plant context when they are operated?
top_tags = merged['Source'].value_counts().head(8).index.tolist()

# Create a summary: for each top tag, what is the median plant state when it's operated?
tag_context_summary = []
for tag in top_tags:
    tag_data = merged[merged['Source'] == tag]
    row = {
        'Tag': tag,
        'N_actions': len(tag_data),
        'target_norm_pos_median': tag_data['merged_ctx_03LIC_1071_norm_pos'].median(),
        'alarm_proximity_median': tag_data['merged_ctx_alarm_proximity'].median(),
        'target_3m_delta_median': tag_data['merged_ctx_03LIC_1071_local_3m_delta_norm'].median(),
        'target_5m_delta_median': tag_data['merged_ctx_03LIC_1071_local_5m_delta_norm'].median(),
        'target_episode_roc_median': tag_data['merged_ctx_03LIC_1071_episode_norm_roc'].median(),
        'pct_increase': (tag_data['merged_action_direction'] == 'up').mean() * 100,
        'pct_decrease': (tag_data['merged_action_direction'] == 'down').mean() * 100,
        'median_abs_step': tag_data['merged_step'].abs().median(),
        'pct_before_alarm': (tag_data['merged_action_timing'] == 'before').mean() * 100,
        'pct_during_alarm': (tag_data['merged_action_timing'] == 'during').mean() * 100,
    }
    tag_context_summary.append(row)

tag_ctx_df = pd.DataFrame(tag_context_summary)
print("=== Plant Context Summary When Each Tag Is Operated ===")
print(tag_ctx_df.to_string(index=False, float_format='%.3f'))

=== Plant Context Summary When Each Tag Is Operated ===
       Tag  N_actions  target_norm_pos_median  alarm_proximity_median  target_3m_delta_median  target_5m_delta_median  target_episode_roc_median  pct_increase  pct_decrease  median_abs_step  pct_before_alarm  pct_during_alarm
03FIC_3435       1711                   0.020                   0.486                  -0.024                  -0.030                     -0.204        61.894        37.989            2.000            42.373            34.717
03HIC_1151        873                  -0.191                   0.375                  -0.051                  -0.047                     -0.342        42.497        57.159            2.000            32.188            51.088
03HIC_3100        670                  -0.339                   0.298                  -0.011                  -0.030                     -0.328        31.940        67.910            4.000            29.552            43.731
03PIC_1013        653                   

In [8]:
# Action direction consistency: For each tag, is the direction always the same or does it vary with context?
# A tag with high direction consistency suggests a "fixed playbook"
direction_consistency = []
for tag in top_tags:
    tag_data = merged[merged['Source'] == tag].dropna(subset=['merged_action_direction'])
    if len(tag_data) == 0:
        continue
    dir_counts = tag_data['merged_action_direction'].value_counts()
    dominant_pct = dir_counts.iloc[0] / len(tag_data) * 100
    dominant_dir = dir_counts.index[0]
    direction_consistency.append({
        'Tag': tag,
        'N': len(tag_data),
        'dominant_direction': dominant_dir,
        'dominant_pct': dominant_pct,
        'consistency': 'High' if dominant_pct > 80 else ('Medium' if dominant_pct > 60 else 'Mixed')
    })

dir_df = pd.DataFrame(direction_consistency)
print("=== Action Direction Consistency by Tag ===")
print("(High = >80% same direction, Medium = 60-80%, Mixed = <60%)\n")
print(dir_df.to_string(index=False, float_format='%.1f'))
print("\nInsight: Tags with HIGH consistency have a fixed directional response.")
print("Tags with MIXED consistency likely depend on specific plant context to determine direction.")

=== Action Direction Consistency by Tag ===
(High = >80% same direction, Medium = 60-80%, Mixed = <60%)

       Tag    N dominant_direction  dominant_pct consistency
03FIC_3435 1711                 up          61.9      Medium
03HIC_1151  873               down          57.2       Mixed
03HIC_3100  670               down          67.9      Medium
03PIC_1013  653               down          54.1       Mixed
03LIC_1071  553                 up          60.0      Medium
03LIC_1034  527                 up          50.3       Mixed
03HIC_1141  411               down          69.8      Medium
03HIC_3132  305                 up          72.1      Medium

Insight: Tags with HIGH consistency have a fixed directional response.
Tags with MIXED consistency likely depend on specific plant context to determine direction.


In [9]:
# For tags with MIXED direction: does plant context explain when operators increase vs decrease?
# Compare context features between increase vs decrease actions for each mixed tag
mixed_tags = dir_df[dir_df['consistency'] == 'Mixed']['Tag'].tolist()
if not mixed_tags:
    mixed_tags = dir_df[dir_df['consistency'] != 'High']['Tag'].tolist()

print("=== Context Comparison: Increase vs Decrease for Mixed-Direction Tags ===\n")
for tag in mixed_tags[:4]:  # Top 4
    tag_data = merged[(merged['Source'] == tag) & merged['merged_action_direction'].isin(['up', 'down'])].copy()
    if len(tag_data) < 10:
        continue
    
    # Compare key context features
    compare_cols = [
        'merged_ctx_03LIC_1071_norm_pos',
        'merged_ctx_alarm_proximity',
        'merged_ctx_03LIC_1071_local_3m_delta_norm',
        'merged_ctx_03LIC_1071_episode_norm_roc'
    ]
    
    inc = tag_data[tag_data['merged_action_direction'] == 'up']
    dec = tag_data[tag_data['merged_action_direction'] == 'down']
    
    print(f"--- {tag} (↑{len(inc)} / ↓{len(dec)}) ---")
    for col in compare_cols:
        short_name = col.replace('merged_ctx_', '').replace('03LIC_1071_', '')
        inc_med = inc[col].median()
        dec_med = dec[col].median()
        diff = inc_med - dec_med
        print(f"  {short_name:30s}  ↑median={inc_med:+.3f}  ↓median={dec_med:+.3f}  (diff={diff:+.3f})")
    print()

=== Context Comparison: Increase vs Decrease for Mixed-Direction Tags ===

--- 03HIC_1151 (↑371 / ↓499) ---
  norm_pos                        ↑median=+0.029  ↓median=-0.323  (diff=+0.352)
  alarm_proximity                 ↑median=+0.491  ↓median=+0.306  (diff=+0.185)
  local_3m_delta_norm             ↑median=-0.158  ↓median=-0.015  (diff=-0.143)
  episode_norm_roc                ↑median=-0.001  ↓median=-0.513  (diff=+0.512)

--- 03PIC_1013 (↑292 / ↓353) ---
  norm_pos                        ↑median=+0.537  ↓median=+0.175  (diff=+0.363)
  alarm_proximity                 ↑median=+0.757  ↓median=+0.567  (diff=+0.190)
  local_3m_delta_norm             ↑median=-0.000  ↓median=-0.070  (diff=+0.070)
  episode_norm_roc                ↑median=-0.125  ↓median=-0.442  (diff=+0.316)

--- 03LIC_1034 (↑265 / ↓262) ---
  norm_pos                        ↑median=+0.212  ↓median=+0.149  (diff=+0.063)
  alarm_proximity                 ↑median=+0.587  ↓median=+0.554  (diff=+0.033)
  local_3m_delta_norm   

In [10]:
# Does action magnitude correlate with severity?
# Hypothesis: Operators apply larger steps when the situation is more critical
merged_with_ctx = merged.dropna(subset=['merged_ctx_alarm_proximity', 'merged_step']).copy()
merged_with_ctx['abs_step'] = merged_with_ctx['merged_step'].abs()

# Bin alarm proximity into severity levels
merged_with_ctx['severity'] = pd.cut(
    merged_with_ctx['merged_ctx_alarm_proximity'],
    bins=[-np.inf, -0.5, 0, 0.5, 1.0, np.inf],
    labels=['Deep alarm (<-0.5)', 'In alarm (-0.5 to 0)', 'Near threshold (0 to 0.5)', 
            'Safe (0.5 to 1.0)', 'Well safe (>1.0)']
)

fig = px.box(
    merged_with_ctx, x='severity', y='abs_step', 
    color='severity',
    title='Action Magnitude vs Alarm Severity',
    labels={'abs_step': 'Absolute Step Size', 'severity': 'Alarm Proximity Level'}
)
fig.update_layout(height=450, showlegend=False, xaxis_tickangle=-20)
fig.show()

print("\nMedian absolute step by severity:")
print(merged_with_ctx.groupby('severity')['abs_step'].agg(['median', 'mean', 'count']).to_string())


Median absolute step by severity:
                           median      mean  count
severity                                          
Deep alarm (<-0.5)            2.0  5.047218   1104
In alarm (-0.5 to 0)          2.0  4.834296    777
Near threshold (0 to 0.5)     2.0  4.419430   2083
Safe (0.5 to 1.0)             2.0  3.799670   1637
Well safe (>1.0)              2.0  4.341849   1643


In [11]:
# Does the rate of decline influence which tag gets operated?
# Create scatter: for each action, show target's 5m delta vs which tag was chosen
top6_tags = merged['Source'].value_counts().head(6).index.tolist()
plot_data = merged[merged['Source'].isin(top6_tags)].dropna(
    subset=['merged_ctx_03LIC_1071_local_5m_delta_norm', 'merged_ctx_alarm_proximity']
)

fig = px.scatter(
    plot_data,
    x='merged_ctx_alarm_proximity',
    y='merged_ctx_03LIC_1071_local_5m_delta_norm',
    color='Source',
    opacity=0.5,
    title='Plant State When Each Tag Is Operated (Target PV Position vs Momentum)',
    labels={
        'merged_ctx_alarm_proximity': 'Alarm Proximity (+ = safe, - = in alarm)',
        'merged_ctx_03LIC_1071_local_5m_delta_norm': 'Target 5-min Momentum (+ = rising, - = falling)'
    }
)
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Threshold")
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(height=500)
fig.show()

print("Insight: If tags occupy distinct regions in this space, it suggests operators")
print("choose WHICH tag to operate based on the plant state (proximity + momentum).")

Insight: If tags occupy distinct regions in this space, it suggests operators
choose WHICH tag to operate based on the plant state (proximity + momentum).


## 3. Clustering Plant States at Action Time

Now let's use unsupervised clustering to discover natural groupings of plant states when actions are taken. If distinct clusters emerge, they represent different "situations" that operators face — and we can then check if specific actions map to specific clusters.

In [12]:
# Use a focused set of features for clustering:
# - Target tag position, momentum, and alarm proximity
# - Key related tags' positions and momenta (use norm_pos + 5m delta for top related tags)
# This avoids the "curse of dimensionality" while capturing meaningful variation

# Select clustering features
cluster_features = (
    ['merged_ctx_alarm_proximity', 'merged_ctx_time_progress_ratio'] +
    [f'merged_ctx_{t}_norm_pos' for t in ['03LIC_1071', '03PIC_1013', '03FIC_3415', 
                                           '03LIC_1016', '03LIC_1085', '03TIC_1142']] +
    [f'merged_ctx_{t}_local_5m_delta_norm' for t in ['03LIC_1071', '03PIC_1013', '03FIC_3415',
                                                      '03LIC_1016', '03LIC_1085', '03TIC_1142']] +
    ['merged_ctx_03LIC_1071_episode_norm_roc']
)

# Prepare data - drop rows with missing context
cluster_data = merged[cluster_features + ['Source', 'merged_action_direction', 'merged_step']].dropna(subset=cluster_features)
print(f"Actions with complete context for clustering: {len(cluster_data)} / {len(merged)}")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data[cluster_features])

# Determine optimal K using elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig = px.line(x=list(K_range), y=inertias, markers=True,
              title='Elbow Method: Optimal Number of Plant State Clusters',
              labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'})
fig.update_layout(height=350)
fig.show()

Actions with complete context for clustering: 6197 / 7246


In [13]:
# Apply KMeans with K=5 (reasonable for interpretability)
K = 3
km = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_data['plant_state_cluster'] = km.fit_predict(X_scaled)

# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
cluster_data['pca1'] = X_pca[:, 0]
cluster_data['pca2'] = X_pca[:, 1]

fig = px.scatter(
    cluster_data, x='pca1', y='pca2', 
    color=cluster_data['plant_state_cluster'].astype(str),
    opacity=0.5,
    title=f'Plant State Clusters (K={K}) — PCA Projection',
    labels={'pca1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)',
            'pca2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)',
            'color': 'Cluster'}
)
fig.update_layout(height=500)
fig.show()

print(f"PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
print(f"\nCluster sizes:")
print(cluster_data['plant_state_cluster'].value_counts().sort_index().to_string())

PCA explained variance: PC1=22.7%, PC2=11.3%

Cluster sizes:
plant_state_cluster
0    4337
1    1097
2     763


In [14]:
# Characterize each plant state cluster
print("=== Plant State Cluster Profiles ===\n")
cluster_profiles = cluster_data.groupby('plant_state_cluster')[cluster_features].median()

# Rename features for readability
readable_names = {
    'merged_ctx_alarm_proximity': 'alarm_proximity',
    'merged_ctx_time_progress_ratio': 'time_progress',
    'merged_ctx_03LIC_1071_norm_pos': '1071_position',
    'merged_ctx_03PIC_1013_norm_pos': 'PIC1013_position',
    'merged_ctx_03FIC_3415_norm_pos': 'FIC3415_position',
    'merged_ctx_03LIC_1016_norm_pos': '1016_position',
    'merged_ctx_03LIC_1085_norm_pos': '1085_position',
    'merged_ctx_03TIC_1142_norm_pos': 'TIC1142_position',
    'merged_ctx_03LIC_1071_local_5m_delta_norm': '1071_momentum',
    'merged_ctx_03PIC_1013_local_5m_delta_norm': 'PIC1013_momentum',
    'merged_ctx_03FIC_3415_local_5m_delta_norm': 'FIC3415_momentum',
    'merged_ctx_03LIC_1016_local_5m_delta_norm': '1016_momentum',
    'merged_ctx_03LIC_1085_local_5m_delta_norm': '1085_momentum',
    'merged_ctx_03TIC_1142_local_5m_delta_norm': 'TIC1142_momentum',
    'merged_ctx_03LIC_1071_episode_norm_roc': '1071_episode_roc',
}
cluster_profiles = cluster_profiles.rename(columns=readable_names)
print(cluster_profiles.T.to_string(float_format='%.3f'))

# Interpret clusters
print("\n\n=== Cluster Interpretation ===")
for c in range(K):
    profile = cluster_profiles.loc[c]
    alarm_prox = profile['alarm_proximity']
    momentum = profile['1071_momentum']
    position = profile['1071_position']
    
    desc = f"Cluster {c}: "
    if alarm_prox < -0.3:
        desc += "DEEP IN ALARM | "
    elif alarm_prox < 0:
        desc += "JUST BELOW THRESHOLD | "
    elif alarm_prox < 0.5:
        desc += "NEAR THRESHOLD (safe side) | "
    else:
        desc += "WELL ABOVE THRESHOLD | "
    
    if momentum < -0.1:
        desc += "DECLINING rapidly"
    elif momentum < 0:
        desc += "declining slowly"
    elif momentum > 0.1:
        desc += "RECOVERING"
    else:
        desc += "stable"
    
    print(f"  {desc}")

=== Plant State Cluster Profiles ===

plant_state_cluster      0      1      2
alarm_proximity      0.484 -1.696  2.272
time_progress        2.108  3.840  2.803
1071_position        0.017 -4.141  3.425
PIC1013_position     0.620  1.934  1.934
FIC3415_position     0.289 -1.135  0.164
1016_position        0.285 -0.848  1.642
1085_position        0.532  0.283  0.601
TIC1142_position     3.346 14.961  7.491
1071_momentum       -0.090 -0.363  2.406
PIC1013_momentum     0.000  0.000 -0.000
FIC3415_momentum     0.004  0.004  0.000
1016_momentum       -0.042 -0.048  1.094
1085_momentum        0.018 -0.055  0.188
TIC1142_momentum    -0.019 -0.145  0.000
1071_episode_roc    -0.177 -4.277  2.652


=== Cluster Interpretation ===
  Cluster 0: NEAR THRESHOLD (safe side) | declining slowly
  Cluster 1: DEEP IN ALARM | DECLINING rapidly
  Cluster 2: WELL ABOVE THRESHOLD | RECOVERING


In [15]:
# KEY QUESTION: Do operators take different actions in different plant state clusters?
print("=== Action Patterns Per Plant State Cluster ===\n")

for c in range(K):
    c_data = cluster_data[cluster_data['plant_state_cluster'] == c]
    print(f"--- Cluster {c} ({len(c_data)} actions) ---")
    
    # Top tags operated
    top3 = c_data['Source'].value_counts().head(5)
    total = len(c_data)
    print(f"  Top tags: {', '.join([f'{t} ({100*n/total:.0f}%)' for t, n in top3.items()])}")
    
    # Direction distribution
    dir_dist = c_data['merged_action_direction'].value_counts()
    print(f"  Direction: {', '.join([f'{d}: {100*n/total:.0f}%' for d, n in dir_dist.items()])}")
    
    # Median step
    med_step = c_data['merged_step'].median()
    print(f"  Median step: {med_step:+.2f}")
    print()

=== Action Patterns Per Plant State Cluster ===

--- Cluster 0 (4337 actions) ---
  Top tags: 03FIC_3435 (30%), 03HIC_1151 (11%), 03LIC_1034 (10%), 03PIC_1013 (10%), 03HIC_3100 (9%)
  Direction: up: 53%, down: 47%, none: 0%
  Median step: +0.50

--- Cluster 1 (1097 actions) ---
  Top tags: 03HIC_1151 (18%), 03LIC_1071 (16%), 03HIC_1141 (12%), 03LIC_1016 (11%), 03PIC_1013 (10%)
  Direction: down: 54%, up: 45%, none: 1%
  Median step: -0.50

--- Cluster 2 (763 actions) ---
  Top tags: 03LIC_1071 (22%), 03HIC_1151 (18%), 03PIC_1013 (13%), 03FIC_3435 (11%), 03LIC_1016 (8%)
  Direction: down: 53%, up: 46%, none: 1%
  Median step: -1.00



In [16]:
# Visualize: Heatmap of tag operation frequency by plant state cluster
# Shows if certain tags are preferred in certain situations
top10_tags = merged['Source'].value_counts().head(10).index.tolist()
cluster_tag_matrix = pd.crosstab(
    cluster_data[cluster_data['Source'].isin(top10_tags)]['plant_state_cluster'],
    cluster_data[cluster_data['Source'].isin(top10_tags)]['Source'],
    normalize='index'
) * 100

fig = px.imshow(
    cluster_tag_matrix.T,
    text_auto='.0f',
    color_continuous_scale='Blues',
    title='Tag Operation Frequency (%) by Plant State Cluster',
    labels={'x': 'Plant State Cluster', 'y': 'Tag Operated', 'color': '% of actions'}
)
fig.update_layout(height=500)
fig.show()

print("Insight: Columns with varying color patterns indicate that tag choice DEPENDS on plant state.")
print("Uniform columns suggest the tag is operated regardless of state.")

Insight: Columns with varying color patterns indicate that tag choice DEPENDS on plant state.
Uniform columns suggest the tag is operated regardless of state.


## 4. Tag-Specific Action Triggers

For the most frequently operated tags, what specific plant conditions trigger their operation? 
This helps answer: "Given this plant state, which tag should I operate and in which direction?"

In [16]:
# For each top tag: what's unique about the plant state when it's operated vs when it's NOT?
# Use the FULL set of norm_pos features to find which related tags are in unusual positions
# when a specific tag gets operated

top5_tags = merged['Source'].value_counts().head(5).index.tolist()

# For each operated tag, compute median norm_pos of ALL context tags at action time
# Then compare to the overall median
overall_median_pos = merged[norm_pos_cols].median()

print("=== Distinctive Plant Signatures When Each Tag Is Operated ===")
print("(Shows which context tags are MOST different from their overall median)\n")

for tag in top5_tags:
    tag_data = merged[merged['Source'] == tag]
    tag_median_pos = tag_data[norm_pos_cols].median()
    
    # Difference from overall
    diff = tag_median_pos - overall_median_pos
    diff.index = [c.replace('merged_ctx_', '').replace('_norm_pos', '') for c in diff.index]
    
    # Top 5 most distinctive (largest absolute difference)
    top_diffs = diff.abs().nlargest(5)
    print(f"--- {tag} ({len(tag_data)} actions) ---")
    for ctx_tag in top_diffs.index:
        d = diff[ctx_tag]
        direction = "↑ HIGH" if d > 0 else "↓ LOW"
        print(f"  {ctx_tag:20s}: {direction} (Δ={d:+.3f})")
    print()

=== Distinctive Plant Signatures When Each Tag Is Operated ===
(Shows which context tags are MOST different from their overall median)

--- 03FIC_3435 (1711 actions) ---
  03TIC_1142          : ↓ LOW (Δ=-2.232)
  03FI_1141A          : ↓ LOW (Δ=-1.280)
  03TI_1081           : ↓ LOW (Δ=-1.205)
  03TI_1901           : ↓ LOW (Δ=-1.202)
  03PIC_3131          : ↑ HIGH (Δ=+1.053)

--- 03HIC_1151 (873 actions) ---
  03FI_1141A          : ↓ LOW (Δ=-61555.058)
  03TI_1081           : ↑ HIGH (Δ=+2.581)
  03TIC_1142          : ↑ HIGH (Δ=+2.459)
  03PI_1141A          : ↑ HIGH (Δ=+2.192)
  03TI_1901           : ↑ HIGH (Δ=+1.958)

--- 03HIC_3100 (670 actions) ---
  03FI_1141A          : ↑ HIGH (Δ=+6837.388)
  03TIC_1142          : ↑ HIGH (Δ=+2.189)
  02FI_1000           : ↓ LOW (Δ=-1.635)
  03PIC_3131          : ↑ HIGH (Δ=+1.630)
  03PIC_1068          : ↑ HIGH (Δ=+1.064)

--- 03PIC_1013 (653 actions) ---
  03FI_1141A          : ↓ LOW (Δ=-6840.589)
  03TI_1081           : ↑ HIGH (Δ=+4.115)
  03PI_1141

In [17]:
# Radar chart: Compare plant context profiles for top 5 operated tags
# Use norm_pos features to show "fingerprint" of each tag's typical action context

# Select a subset of readable tags for the radar
radar_features = norm_pos_cols[:8]  # First 8 tags for readability
radar_labels = [c.replace('merged_ctx_', '').replace('_norm_pos', '') for c in radar_features]

fig = go.Figure()
for tag in top5_tags:
    tag_data = merged[merged['Source'] == tag]
    values = tag_data[radar_features].median().values.tolist()
    values.append(values[0])  # Close the polygon
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_labels + [radar_labels[0]],
        fill='toself',
        name=tag,
        opacity=0.6
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Plant Context "Fingerprint" for Top 5 Operated Tags<br>(Normalized Position of Context Tags at Action Time)',
    height=550
)
fig.show()

print("Insight: If these fingerprints are distinct, operators are choosing tags based on")
print("the broader plant state, not just the target tag alone.")

Insight: If these fingerprints are distinct, operators are choosing tags based on
the broader plant state, not just the target tag alone.


In [18]:
# Feature importance: Which context features best predict action DIRECTION?
# Use a simple approach: for each feature, compute the separation between increase vs decrease actions
from scipy import stats

merged_dir = merged[merged['merged_action_direction'].isin(['up', 'down'])].copy()
merged_dir['dir_binary'] = (merged_dir['merged_action_direction'] == 'up').astype(int)

# Compute point-biserial correlation between each context feature and direction
feature_importance = []
all_ctx_features = norm_pos_cols + local_5m_cols + ['merged_ctx_alarm_proximity', 'merged_ctx_03LIC_1071_episode_norm_roc']

for feat in all_ctx_features:
    valid = merged_dir[[feat, 'dir_binary']].dropna()
    if len(valid) < 30:
        continue
    corr, pval = stats.pointbiserialr(valid['dir_binary'], valid[feat])
    feature_importance.append({
        'feature': feat.replace('merged_ctx_', ''),
        'correlation_with_increase': corr,
        'abs_correlation': abs(corr),
        'p_value': pval
    })

fi_df = pd.DataFrame(feature_importance).sort_values('abs_correlation', ascending=False)

print("=== Top 15 Features Most Predictive of Action Direction ===")
print("(Positive correlation = feature HIGH → operator INCREASES; Negative = feature HIGH → operator DECREASES)\n")
print(fi_df.head(15).to_string(index=False, float_format='%.4f'))

=== Top 15 Features Most Predictive of Action Direction ===
(Positive correlation = feature HIGH → operator INCREASES; Negative = feature HIGH → operator DECREASES)

                       feature  correlation_with_increase  abs_correlation  p_value
            03TI_1015_norm_pos                    -0.1076           0.1076   0.0000
           03PIC_1013_norm_pos                    -0.1014           0.1014   0.0000
           03PIC_1104_norm_pos                     0.0955           0.0955   0.0000
           03PI_1141A_norm_pos                    -0.0953           0.0953   0.0000
            03PI_1495_norm_pos                    -0.0951           0.0951   0.0000
            03TI_1421_norm_pos                    -0.0834           0.0834   0.0000
            03FI_1151_norm_pos                    -0.0795           0.0795   0.0000
           03PIC_1068_norm_pos                     0.0756           0.0756   0.0000
            03TI_1081_norm_pos                    -0.0701           0.0701   0

In [19]:
# Visualize top feature importances
fig = px.bar(
    fi_df.head(15),
    x='correlation_with_increase', y='feature',
    orientation='h', color='correlation_with_increase',
    color_continuous_scale='RdBu', color_continuous_midpoint=0,
    title='Context Features Most Predictive of Action Direction (Increase vs Decrease)',
    labels={'correlation_with_increase': 'Correlation with Increase Direction', 'feature': ''}
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

## 5. Action Sequencing: Do Operators Follow Fixed "Playbooks"?

Let's look at whether operators follow recurring sequences of actions within alarm clusters. A "playbook" would be: first operate tag A, then tag B, then tag C — consistently across similar situations.

In [20]:
# Extract action sequences per cluster (order of tags operated)
# Focus on unique merged actions (SINGLE/START) within each cluster
seq_data = merged.sort_values(['cluster_id', 'merged_action_timestamp']).copy()

# Build sequence strings per cluster: tag_direction pairs in order
cluster_sequences = {}
for cid, grp in seq_data.groupby('cluster_id'):
    seq = []
    for _, row in grp.iterrows():
        tag = row['Source']
        direction = row['merged_action_direction']
        if pd.notna(direction):
            seq.append(f"{tag}_{direction}")
    if seq:
        cluster_sequences[cid] = seq

# Find most common first actions
first_actions = [seq[0] for seq in cluster_sequences.values() if len(seq) > 0]
print("=== Most Common FIRST Action in Alarm Clusters ===")
first_action_counts = pd.Series(first_actions).value_counts().head(10)
print(first_action_counts.to_string())
print(f"\nTotal clusters with actions: {len(cluster_sequences)}")

# Most common 2-action pairs (bigrams)
bigrams = []
for seq in cluster_sequences.values():
    for i in range(len(seq) - 1):
        bigrams.append(f"{seq[i]} → {seq[i+1]}")

print("\n\n=== Most Common Action Pairs (Sequential) ===")
bigram_counts = pd.Series(bigrams).value_counts().head(15)
print(bigram_counts.to_string())

=== Most Common FIRST Action in Alarm Clusters ===
03FIC_3435_down    72
03FIC_3435_up      58
03PIC_1013_down    44
03HIC_1151_up      36
03LIC_1034_down    32
03LIC_1034_up      20
03LIC_1071_up      19
03PIC_1013_up      19
03HIC_1151_down    18
03HIC_3132_up      11

Total clusters with actions: 419


=== Most Common Action Pairs (Sequential) ===
03FIC_3435_up → 03FIC_3435_up        553
03FIC_3435_down → 03FIC_3435_down    291
03LIC_1034_up → 03LIC_1034_up        169
03HIC_1151_down → 03HIC_1151_down    169
03PIC_1013_down → 03PIC_1013_down    145
03LIC_1034_down → 03LIC_1034_down    141
03HIC_3100_down → 03HIC_3100_down    139
03HIC_1151_up → 03HIC_1151_up        121
03LIC_1071_up → 03LIC_1071_up        118
03PIC_1013_up → 03PIC_1013_up        116
03LIC_1071_down → 03LIC_1071_down     88
03FIC_3435_up → 03FIC_3435_down       68
03HIC_1141_down → 03HIC_1151_down     68
03FIC_3435_down → 03FIC_3435_up       62
03FIC_3435_up → 03HIC_3100_down       61


In [21]:
# Do the action sequences differ by plant state at the START of the cluster?
# For each cluster, determine the plant state (from the first action's context)
first_action_per_cluster = seq_data.groupby('cluster_id').first().reset_index()

# Merge with cluster assignment (if the first action was in our clustering data)
first_with_cluster = first_action_per_cluster.merge(
    cluster_data[['plant_state_cluster']].reset_index(drop=False),
    left_index=True, right_index=True, how='inner'
)

# For clusters where we have plant state: what's the first action?
if len(first_with_cluster) > 0:
    # Use a simpler approach: attach plant state to first action per cluster
    first_actions_ctx = []
    for cid, grp in seq_data.groupby('cluster_id'):
        first_row = grp.iloc[0]
        if cid in cluster_data['cluster_id'].values if 'cluster_id' in cluster_data.columns else False:
            continue
        first_actions_ctx.append({
            'cluster_id': cid,
            'first_tag': first_row['Source'],
            'first_direction': first_row['merged_action_direction'],
            'alarm_proximity': first_row['merged_ctx_alarm_proximity'],
            'target_momentum': first_row['merged_ctx_03LIC_1071_local_5m_delta_norm']
        })

# Instead, directly analyze: what first action is taken depending on alarm proximity
first_per_cluster = seq_data.groupby('cluster_id').first().reset_index()
first_per_cluster['proximity_bin'] = pd.cut(
    first_per_cluster['merged_ctx_alarm_proximity'],
    bins=[-np.inf, -0.3, 0, 0.5, 1.0, np.inf],
    labels=['Deep alarm', 'Just below', 'Near threshold', 'Safe', 'Well safe']
)

print("=== First Tag Operated by Alarm Proximity at Start ===\n")
for prox_bin in ['Deep alarm', 'Just below', 'Near threshold', 'Safe', 'Well safe']:
    bin_data = first_per_cluster[first_per_cluster['proximity_bin'] == prox_bin]
    if len(bin_data) == 0:
        continue
    top3 = bin_data['Source'].value_counts().head(3)
    print(f"{prox_bin} ({len(bin_data)} clusters):")
    for tag, count in top3.items():
        pct = 100 * count / len(bin_data)
        print(f"  {tag}: {count} ({pct:.0f}%)")
    print()

=== First Tag Operated by Alarm Proximity at Start ===

Deep alarm (10 clusters):
  03HIC_1151: 2 (20%)
  03PIC_1013: 2 (20%)
  03LIC_1071: 2 (20%)

Just below (21 clusters):
  03LIC_1071: 7 (33%)
  03FIC_3435: 5 (24%)
  03HIC_1151: 4 (19%)

Near threshold (140 clusters):
  03FIC_3435: 46 (33%)
  03LIC_1034: 19 (14%)
  03PIC_1013: 19 (14%)

Safe (146 clusters):
  03FIC_3435: 51 (35%)
  03PIC_1013: 24 (16%)
  03LIC_1034: 23 (16%)

Well safe (102 clusters):
  03FIC_3435: 26 (25%)
  03PIC_1013: 17 (17%)
  03LIC_1071: 13 (13%)



In [22]:
# Analyze: Is there a relationship between the "own tag" context and action on that tag?
# For tags that have their own norm_pos in context, does their position influence action direction?
# E.g., when 03PIC_1013 is HIGH in its range, do operators decrease it?

tags_with_own_context = ['03PIC_1013', '03FIC_3415', '03LIC_1016', '03LIC_1085', 
                         '03LIC_1071', '03TIC_1142', '03TIC_1145', '03PIC_1068',
                         '03PIC_1104', '03PIC_3131', '03FIC_1085', '03LIC_1094',
                         '03LIC_1097', '03LIC_3178']

print("=== Self-Position vs Action Direction ===")
print("(Does the operated tag's own position predict whether operator increases or decreases it?)\n")

self_analysis = []
for tag in tags_with_own_context:
    pos_col = f'merged_ctx_{tag}_norm_pos'
    if pos_col not in merged.columns:
        continue
    
    tag_actions = merged[(merged['Source'] == tag) & merged['merged_action_direction'].isin(['up', 'down'])].copy()
    if len(tag_actions) < 10:
        continue
    
    inc_pos = tag_actions[tag_actions['merged_action_direction'] == 'up'][pos_col].median()
    dec_pos = tag_actions[tag_actions['merged_action_direction'] == 'down'][pos_col].median()
    
    self_analysis.append({
        'Tag': tag,
        'N_actions': len(tag_actions),
        'median_pos_when_increased': inc_pos,
        'median_pos_when_decreased': dec_pos,
        'diff': inc_pos - dec_pos if pd.notna(inc_pos) and pd.notna(dec_pos) else np.nan
    })

self_df = pd.DataFrame(self_analysis).sort_values('diff', key=abs, ascending=False)
print(self_df.to_string(index=False, float_format='%.3f'))
print("\nInsight: Negative diff → tag is LOWER when increased (makes sense: compensating)")
print("Positive diff → tag is HIGHER when increased (unusual: amplifying)")
print("Near zero → own position doesn't determine direction (context-driven)")

=== Self-Position vs Action Direction ===
(Does the operated tag's own position predict whether operator increases or decreases it?)

       Tag  N_actions  median_pos_when_increased  median_pos_when_decreased   diff
03LIC_1016        262                     -2.313                      2.052 -4.364
03LIC_1094         16                      4.841                      1.170  3.671
03LIC_1097         33                     -3.146                      0.359 -3.505
03LIC_1071        549                     -1.142                      2.166 -3.308
03PIC_1068         85                      3.169                      4.494 -1.325
03LIC_1085        140                      0.763                      1.824 -1.061
03FIC_3415        213                      0.997                      0.037  0.960
03PIC_3131        170                      3.119                      2.414  0.705
03FIC_1085         89                      0.434                      0.865 -0.431
03LIC_3178         59               

## 6. Temporal Patterns: How Does Plant Context Evolve During an Episode?

Do operators take different actions early vs late in an alarm episode? Does the urgency (time_progress_ratio) influence the choice?

In [23]:
# Bin actions by time_progress_ratio to see how behavior changes over the episode timeline
merged_temporal = merged.dropna(subset=['merged_ctx_time_progress_ratio']).copy()
merged_temporal['time_bin'] = pd.cut(
    merged_temporal['merged_ctx_time_progress_ratio'],
    bins=[0, 1, 1.5, 2, 2.5, 3, 4, np.inf],
    labels=['0-1 (early)', '1-1.5', '1.5-2', '2-2.5', '2.5-3', '3-4', '4+ (late)']
)

# Action characteristics by temporal phase
temporal_stats = merged_temporal.groupby('time_bin').agg(
    n_actions=('Source', 'count'),
    pct_increase=('merged_action_direction', lambda x: (x == 'up').mean() * 100),
    median_abs_step=('merged_step', lambda x: x.abs().median()),
    top_tag=('Source', lambda x: x.value_counts().index[0] if len(x) > 0 else ''),
    median_alarm_prox=('merged_ctx_alarm_proximity', 'median')
).reset_index()

print("=== Action Characteristics by Episode Timeline Phase ===")
print("(time_progress_ratio > 1 means after alarm start; < 1 means before alarm)\n")
print(temporal_stats.to_string(index=False, float_format='%.2f'))

=== Action Characteristics by Episode Timeline Phase ===
(time_progress_ratio > 1 means after alarm start; < 1 means before alarm)

   time_bin  n_actions  pct_increase  median_abs_step    top_tag  median_alarm_prox
0-1 (early)        309         55.66             4.00 03FIC_3435               0.91
      1-1.5        513         54.78             2.00 03FIC_3435               0.57
      1.5-2       1500         49.47             2.00 03FIC_3435               0.57
      2-2.5       1358         52.36             2.00 03FIC_3435               0.26
      2.5-3       1039         54.09             2.00 03FIC_3435               0.46
        3-4        889         50.39             2.00 03FIC_3435               0.37
  4+ (late)       1020         45.69             2.00 03HIC_1151               0.01


In [24]:
# Visualize: How does the tag choice evolve over the episode timeline?
top6 = merged['Source'].value_counts().head(6).index.tolist()
temporal_tag = merged_temporal[merged_temporal['Source'].isin(top6)].copy()

# Stacked area-like visualization
tag_time_crosstab = pd.crosstab(
    temporal_tag['time_bin'], temporal_tag['Source'], normalize='index'
) * 100

fig = px.bar(
    tag_time_crosstab.reset_index().melt(id_vars='time_bin', var_name='Tag', value_name='Percentage'),
    x='time_bin', y='Percentage', color='Tag',
    title='Tag Choice Evolution Over Episode Timeline (Top 6 Tags)',
    labels={'time_bin': 'Episode Timeline Phase', 'Percentage': '% of Actions'},
    barmode='stack'
)
fig.update_layout(height=450)
fig.show()

print("Insight: If the tag distribution changes significantly across phases,")
print("it suggests a temporal 'playbook' — certain tags are preferred early vs late.")

Insight: If the tag distribution changes significantly across phases,
it suggests a temporal 'playbook' — certain tags are preferred early vs late.


## 7. Co-occurrence Patterns: Which Tags Are Operated Together?

If certain tags are consistently operated together within the same cluster, it suggests a "bundle" of corrective actions that operators view as a package.

In [25]:
# Co-occurrence matrix: which tags appear together in the same cluster?
from itertools import combinations

top12_tags = merged['Source'].value_counts().head(12).index.tolist()

# For each cluster, find which of the top tags were operated
cluster_tag_sets = merged[merged['Source'].isin(top12_tags)].groupby('cluster_id')['Source'].apply(set)

# Build co-occurrence matrix
cooccurrence = pd.DataFrame(0, index=top12_tags, columns=top12_tags)
for tag_set in cluster_tag_sets:
    for t1, t2 in combinations(tag_set, 2):
        if t1 in top12_tags and t2 in top12_tags:
            cooccurrence.loc[t1, t2] += 1
            cooccurrence.loc[t2, t1] += 1

# Normalize by number of clusters each tag appears in
tag_cluster_counts = merged[merged['Source'].isin(top12_tags)].groupby('Source')['cluster_id'].nunique()

# Jaccard similarity: co-occur / (appears_A + appears_B - co-occur)
jaccard = pd.DataFrame(0.0, index=top12_tags, columns=top12_tags)
for t1 in top12_tags:
    for t2 in top12_tags:
        if t1 == t2:
            jaccard.loc[t1, t2] = 1.0
        else:
            union = tag_cluster_counts.get(t1, 0) + tag_cluster_counts.get(t2, 0) - cooccurrence.loc[t1, t2]
            if union > 0:
                jaccard.loc[t1, t2] = cooccurrence.loc[t1, t2] / union

fig = px.imshow(
    jaccard, text_auto='.2f',
    color_continuous_scale='YlOrRd',
    title='Tag Co-occurrence (Jaccard Similarity) Within Alarm Clusters',
    labels={'color': 'Jaccard'}
)
fig.update_layout(height=600, width=700)
fig.show()

print("High Jaccard values indicate tags that are almost always operated TOGETHER.")
print("These may form 'action bundles' that operators apply as a group.")

High Jaccard values indicate tags that are almost always operated TOGETHER.
These may form 'action bundles' that operators apply as a group.


## 8. Summary: Key Findings & Actionable Insights

In [ ]:
# Consolidated summary based on actual results
print("=" * 80)
print("PLANT CONTEXT ANALYSIS — KEY FINDINGS SUMMARY")
print("=" * 80)

print("""
1. FEATURE VALIDATION — Features ARE sensible
   - alarm_proximity correctly separates timing: 'before'/'after' median ≈ +0.62, 
     'during' median = -0.08 (below threshold). ✓
   - Target (03LIC_1071) is in the LOWER half of its range for 67.9% of actions. ✓
   - Target is declining at action time only ~54% of the time (3m and 5m deltas).
     This is barely above 50% — the momentum signal is WEAK at the minute level.
     Operators are NOT always acting while the target is actively falling.

2. DIRECTION CONSISTENCY — NO tag has a truly fixed direction
   - All top 8 tags are 'Mixed' or 'Medium' (50-72% dominant direction).
   - Most consistent: 03HIC_3132 (72% up), 03HIC_1141 (70% down).
   - Least consistent: 03LIC_1034 (50/50), 03PIC_1013 (54% down).
   - CONCLUSION: Direction is NOT fixed for any tag — it depends on context every time.

3. MAGNITUDE vs SEVERITY — NO clear magnitude scaling
   - Median step is 2.0 across ALL severity bins (Deep alarm to Well safe).
   - Mean step shows slight increase in severity (5.0 deep alarm vs 3.8 safe),
     but this is driven by outliers, not a systematic pattern.
   - CONCLUSION: Operators use a fixed step size (2.0) regardless of urgency.

4. PLANT STATE CLUSTERS — 5 distinct states identified, but action overlap is HIGH
   Cluster profiles:
   - C0 (n=527): DEEP IN ALARM (prox=-2.1), stable momentum → HIC_1151, HIC_1141, LIC_1016 (56% down)
   - C1 (n=587): WELL ABOVE threshold (prox=+2.4), RECOVERING (mom=+2.8) → LIC_1071, HIC_1151 (56% down)
   - C2 (n=1987): NEAR threshold safe (prox=+0.6), stable → FIC_3435 dominates (36%), up:56%
   - C3 (n=820): IN ALARM (prox=-0.6), DECLINING rapidly (mom=-2.2) → FIC_3435, LIC_1071, HIC_1151 (52% up)
   - C4 (n=2276): NEAR threshold (prox=+0.5), stable → FIC_3435, PIC_1013, HIC_1151 (50/50)
   
   KEY INSIGHT: 03FIC_3435 dominates in 3 of 5 clusters. The differentiation is moderate —
   only C0 (deep alarm) shows a clearly different tag preference (HIC tags).

5. FEATURE IMPORTANCE FOR DIRECTION — Correlations are VERY WEAK
   - Top predictor: 03TI_1015_norm_pos with |r| = 0.108
   - All correlations are |r| < 0.11 — practically zero predictive power.
   - CONCLUSION: No single plant context feature reliably predicts direction.
     The decision may be multi-variate or rely on information not in these features.

6. SELF-POSITION vs DIRECTION — Some tags show clear self-regulation
   - 03LIC_1016: diff = -4.36 → STRONGLY increased when LOW, decreased when HIGH (compensating). ✓
   - 03LIC_1071: diff = -3.31 → Same pattern (target tag self-regulation). ✓
   - 03LIC_1097: diff = -3.51 → Same compensating pattern. ✓
   - 03FIC_3415: diff = +0.96 → Increased when HIGH (amplifying/different logic).
   - 03PIC_1013: diff = 0.00 → Own position has NO influence on direction.
   
   CONCLUSION: Level controllers (LIC) follow clear self-compensating logic.
   Flow/pressure tags are driven by other factors.

7. TEMPORAL PATTERNS — Limited temporal differentiation
   - 03FIC_3435 is the top tag in ALL time phases EXCEPT '4+ (late)' where HIC_1151 dominates.
   - Early actions (0-1) use larger steps (median=4.0) vs later phases (median=2.0).
   - Direction is ~50/50 in all phases — no clear temporal direction shift.
   - CONCLUSION: Only one temporal rule found: late in episode → switch to HIC_1151.

8. CO-OCCURRENCE — Clear action bundles exist
   - Strongest pair: 03HIC_3100 + 03HIC_3132 (Jaccard=0.49) — these are ALWAYS together.
   - Pair: 03LIC_3153 + 03PIC_3131 (Jaccard=0.49) — another tight bundle.
   - Pair: 03FIC_3435 + 03HIC_1151 (Jaccard=0.39) — most frequent combo (113 clusters).
   - Pair: 03LIC_1016 + 03LIC_1071 (Jaccard=0.35) — level controllers bundled.
   - CONCLUSION: Operators do have "bundles" — these are the reliable patterns.

9. FIRST ACTION BY SEVERITY
   - Deep alarm: HIC_1151 or PIC_1013 (20% each) — different from normal.
   - Just below threshold: LIC_1071 (33%) — direct self-correction first.
   - Near threshold / Safe / Well safe: FIC_3435 dominates (25-35%) — preventive favorite.
   - CONCLUSION: First tag choice DOES shift with severity, especially in deep alarm.
""")

print("=" * 80)
print("\nOVERALL VERDICT:")
print("-" * 80)
print("""
The plant context features are VALID (correctly capture plant state) but have
LIMITED PREDICTIVE POWER for individual action decisions:
- No single feature predicts direction (all |r| < 0.11)
- Magnitude is NOT scaled to severity (always step=2)
- No tag has a fixed direction — all are context-dependent

WHAT DOES WORK:
1. Co-occurrence bundles are real and consistent (HIC_3100+HIC_3132, etc.)
2. Level controllers (LIC) self-regulate based on own position
3. First action choice shifts with severity (deep alarm → HIC tags)
4. Late-episode switch to HIC_1151 is a real temporal pattern
5. 03FIC_3435 is the universal "go-to" tag for most situations

IMPLICATION: The operator decision process is likely based on factors NOT fully
captured in these features (e.g., experience, upstream conditions, verbal handoff)
OR is a complex non-linear multi-variate decision that simple correlations miss.
""")
print("=" * 80)

PLANT CONTEXT ANALYSIS — KEY FINDINGS SUMMARY

1. FEATURE VALIDATION
   - Plant context features are sensible: alarm_proximity correctly distinguishes
     before/during timing, target norm_pos is low during alarm situations,
     and short-term deltas show declining target PV when actions are taken.

2. ACTION DIRECTION CONSISTENCY
   - Some tags (e.g., those with >80% same direction) have FIXED directional 
     responses regardless of context — these are "always increase" or "always decrease" tags.
   - Tags with MIXED direction are context-dependent — the plant state determines
     whether to increase or decrease.

3. MAGNITUDE vs SEVERITY
   - Operators do/do not scale their step sizes based on alarm proximity.
     (Examine the box plot above for the relationship.)

4. PLANT STATE CLUSTERS
   - There are distinct plant states (clusters) when operators act.
   - Different clusters have different preferred tags and directions.
   - This confirms that operators DO follow different 